In [1]:
#### ETL処理
## 
## Step1：発注数と最新単価
import pandas as pd
import sqlite3

In [9]:
# データベースへの接続を開く
conn = sqlite3.connect('factory.db')
cursor = conn.cursor()

In [12]:
# テーブル作成
sql_create_tables = """
-- 既存テーブルの削除
DROP TABLE IF EXISTS m_item;
DROP TABLE IF EXISTS t_inventory;

-- 1. 品目マスタ
CREATE TABLE m_item (
    item_code VARCHAR(20) NOT NULL,
    item_name VARCHAR(100),
    category  VARCHAR(20),
    PRIMARY KEY (item_code)
);

-- 2. 在庫テーブル
CREATE TABLE t_inventory (
    item_code     VARCHAR(20) NOT NULL,
    location_code VARCHAR(20) NOT NULL,
    quantity      INTEGER DEFAULT 0,
    PRIMARY KEY (item_code, location_code),
    FOREIGN KEY (item_code) REFERENCES m_item(item_code)
);
"""

try:
    cursor.executescript(sql_create_tables)
    print("✅ m_item, t_inventory テーブルを作成しました。")
except Exception as e:
    print(f"❌ エラー: {e}")

✅ m_item, t_inventory テーブルを作成しました。


In [13]:
# テーブル作成２
sql_create_bom = """
DROP TABLE IF EXISTS m_bom;

CREATE TABLE m_bom (
    parent_item_code VARCHAR(20) NOT NULL,
    child_item_code  VARCHAR(20) NOT NULL,
    quantity         INTEGER,
    
    PRIMARY KEY (parent_item_code, child_item_code),
    FOREIGN KEY (parent_item_code) REFERENCES m_item(item_code),
    FOREIGN KEY (child_item_code)  REFERENCES m_item(item_code)
);
"""

try:
    cursor.executescript(sql_create_bom)
    print("✅ m_bom テーブルを作成しました。")
except Exception as e:
    print(f"❌ エラー: {e}")

✅ m_bom テーブルを作成しました。


In [26]:
# CSV読み込み
required_files = {
    'm_item.csv': 'm_item',
    'm_bom.csv': 'm_bom',
    'm_purchase_price.csv': 'm_purchase_price',
    't_inventory.csv': 't_inventory'
}

for filename in required_files.keys():
    tablename = required_files[filename]

    # 読み込み
    df = pd.read_csv(filename)

    # DBに保存
    df.to_sql(tablename, conn, if_exists='replace', index=False)

    print(f"{tablename}にデータを登録しました。")

m_itemにデータを登録しました。
m_bomにデータを登録しました。
m_purchase_priceにデータを登録しました。
t_inventoryにデータを登録しました。


In [31]:
# 登録確認
findsql = """
    SELECT * FROM m_item
"""
df_find = pd.read_sql(findsql, conn)
display(df_find)

findsql = """
    SELECT * FROM m_bom
"""
df_find = pd.read_sql(findsql, conn)
display(df_find)

findsql = """
    SELECT * FROM m_purchase_price
"""
df_find = pd.read_sql(findsql, conn)
display(df_find)

findsql = """
    SELECT * FROM t_inventory
"""
df_find = pd.read_sql(findsql, conn)
display(df_find)

,item_code,item_name,category
0,ROBOT-X,Home Cleaning Robot X,Product
1,UNIT-ARM-R,Right Arm Unit,Sub-assembly
2,UNIT-LEG,Drive Unit,Sub-assembly
3,MTR-DC-001,DC Motor (Standard),Component
4,PCB-MAIN-V1,Main Control PCB,Component
5,SCR-M4-10,M4 Screw (10mm),Component
6,SEN-IR-01,Infrared Sensor,Component
7,CHASSIS-01,Main Chassis,Component


,parent_item_code,child_item_code,quantity
0,ROBOT-X,UNIT-ARM-R,1
1,ROBOT-X,UNIT-LEG,2
2,ROBOT-X,CHASSIS-01,1
3,ROBOT-X,PCB-MAIN-V1,1
4,UNIT-ARM-R,MTR-DC-001,3
5,UNIT-ARM-R,SCR-M4-10,12
6,UNIT-LEG,MTR-DC-001,2
7,UNIT-LEG,SCR-M4-10,8
8,CHASSIS-01,SCR-M4-10,6


,item_code,supplier_code,start_date,end_date,unit_price,moq,lead_time_days
0,SCR-M4-10,SUP-A,2020-01-01,None,10.5,100 pcs,3
1,SCR-M4-10,SUP-B,2023-01-01,None,12.0,1 box(500),1
2,MTR-DC-001,SUP-C,2022-04-01,2023-03-31,2500.0,1,14
3,MTR-DC-001,SUP-C,2023-04-01,None,2800.0,1 pc,14
4,PCB-MAIN-V1,SUP-D,2023-01-01,None,15000.0,5 pcs,30
5,SEN-IR-01,SUP-A,2023-01-01,None,850.0,50,5


,item_code,location_code,quantity
0,ROBOT-X,WH-01,2
1,MTR-DC-001,WH-01,15
2,SCR-M4-10,WH-02,350
3,PCB-MAIN-V1,WH-01,0


In [3]:
# --- SQL 1: 発注数（不足数）の抽出 ---
# "Triple Quote" (""") を使うと、改行を含む長い文字列を書けます
sql_shortage = """
WITH RECURSIVE bom_tree AS (
    SELECT 'ROBOT-X' as root_item, child_item_code, quantity * 10 as required_qty 
    FROM m_bom WHERE parent_item_code = 'ROBOT-X'
    UNION ALL
    SELECT parent.root_item, child.child_item_code, parent.required_qty * child.quantity
    FROM bom_tree parent JOIN m_bom child ON parent.child_item_code = child.parent_item_code
),
grouped_req AS (
    SELECT child_item_code, SUM(required_qty) as total_req_qty 
    FROM bom_tree GROUP BY child_item_code
)
SELECT 
    req.child_item_code AS item_code,
    MAX(req.total_req_qty - COALESCE(inv.quantity, 0), 0) AS shortage_qty
FROM grouped_req req
LEFT JOIN t_inventory inv ON req.child_item_code = inv.item_code
WHERE MAX(req.total_req_qty - COALESCE(inv.quantity, 0), 0) > 0;
"""

In [4]:
# --- SQL 2: 最新単価リストの抽出 (Lecture 18の完成形) ---
sql_price = """
WITH ranked_price AS (
    SELECT 
        item_code, supplier_code, unit_price, moq, lead_time_days,
        ROW_NUMBER() OVER (PARTITION BY item_code, supplier_code ORDER BY start_date DESC) as rn
    FROM m_purchase_price
)
SELECT item_code, supplier_code, unit_price, moq, lead_time_days
FROM ranked_price WHERE rn = 1;
"""

In [14]:
## Step2：Extract（抽出）
# 1. 不足リストをロード
df_shortage = pd.read_sql(sql_shortage, conn)